
# Conversational Evaluation Dataset Generation v1

This notebook creates the **Development Set** and **Held-out Test Set** for the conversational preference-tracking component.

## Why there is no Train / Validation / Test split

The project does not train or fine-tune a model. It uses a pretrained LLM behind an API/adapter boundary.

Therefore:

- **Product catalogue** = retrieval corpus / product database, not training data.
- **Development Set** = prompt/schema/validation development and debugging.
- **Held-out Test Set** = final quantitative evaluation only.

The held-out test set should not be repeatedly used to tune the prompt after final evaluation begins.

## Frozen Preference State

```json
{
  "budget_max": null,
  "brand": null,
  "ram_min_gb": null,
  "storage_min_gb": null,
  "screen_size_max_in": null,
  "gpu_required": null,
  "weight_max_kg": null,
  "usage": null,
  "preference_notes": null
}
```

### Executable hard constraints
- `budget_max`
- `brand`
- `ram_min_gb`
- `storage_min_gb`
- `screen_size_max_in`
- `gpu_required`
- `weight_max_kg`

### Semantic/contextual preferences
- `usage`
- `preference_notes`

The ground truth is created **before** the natural-language utterance. This avoids self-labelling.


In [1]:

from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [2]:

import copy
import json
import math
import random
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd

pd.set_option("display.max_colwidth", 220)
pd.set_option("display.max_columns", 50)


## 1. Paths and dataset size

In [3]:

PROJECT_DIR = Path("/content/drive/MyDrive/UTS Courses/Industrial Project")

PRODUCT_DB_PATH = (
    PROJECT_DIR / "data" / "processed" / "laptop_product_database_v2.parquet"
)

EVAL_DIR = PROJECT_DIR / "data" / "evaluation"
EVAL_DIR.mkdir(parents=True, exist_ok=True)

DEV_DIALOGUES_PATH = EVAL_DIR / "development_dialogues_v1.jsonl"
TEST_DIALOGUES_PATH = EVAL_DIR / "heldout_test_dialogues_v1.jsonl"

DEV_TURNS_PATH = EVAL_DIR / "development_turns_v1.csv"
TEST_TURNS_PATH = EVAL_DIR / "heldout_test_turns_v1.csv"

SCHEMA_PATH = EVAL_DIR / "preference_state_schema_v1.json"
SUMMARY_PATH = EVAL_DIR / "evaluation_dataset_summary_v1.json"

# Semester-scale benchmark.
N_DEV_DIALOGUES = 60
N_TEST_GENERATED_DIALOGUES = 120

DEV_SEED = 32040
TEST_SEED = 25971603

assert PRODUCT_DB_PATH.exists(), f"Product database not found: {PRODUCT_DB_PATH}"

print("Product DB:", PRODUCT_DB_PATH)
print("Evaluation output:", EVAL_DIR)


Product DB: /content/drive/MyDrive/UTS Courses/Industrial Project/data/processed/laptop_product_database_v2.parquet
Evaluation output: /content/drive/MyDrive/UTS Courses/Industrial Project/data/evaluation


## 2. Load the retrieval-ready product database

In [4]:

products = pd.read_parquet(PRODUCT_DB_PATH)

print("Products:", f"{len(products):,}")
print("Columns:", list(products.columns))

display(products.head(10))


## 3. Frozen Preference State schema

In [5]:

STATE_FIELDS = [
    "budget_max",
    "brand",
    "ram_min_gb",
    "storage_min_gb",
    "screen_size_max_in",
    "gpu_required",
    "weight_max_kg",
    "usage",
    "preference_notes",
]

HARD_FIELDS = [
    "budget_max",
    "brand",
    "ram_min_gb",
    "storage_min_gb",
    "screen_size_max_in",
    "gpu_required",
    "weight_max_kg",
]

SEMANTIC_FIELDS = [
    "usage",
    "preference_notes",
]

EMPTY_STATE = {field: None for field in STATE_FIELDS}

CANONICAL_BRANDS = [
    "Acer",
    "Apple",
    "ASUS",
    "Dell",
    "Gateway",
    "Gigabyte",
    "HP",
    "Huawei",
    "Lenovo",
    "LG",
    "Microsoft",
    "MSI",
    "Panasonic",
    "Razer",
    "Samsung",
    "Toshiba",
]

RAM_VALUES = [4, 8, 16, 32, 64]
STORAGE_VALUES = [128, 256, 512, 1024, 2048]
SCREEN_VALUES = [13.0, 14.0, 15.0, 16.0, 17.0]
WEIGHT_VALUES = [1.2, 1.5, 1.8, 2.0, 2.5]
BUDGET_VALUES = [600, 800, 1000, 1200, 1500, 1800, 2000, 2500]

USAGE_VALUES = [
    "business",
    "study",
    "gaming",
    "all_round",
]

# Keep user-facing GPU constraints to a small canonical set.
GPU_VALUES = [
    "Integrated",
    "Intel Iris Xe",
    "AMD Radeon",
    "RTX 3050",
    "RTX 3050 Ti",
    "RTX 3060",
    "RTX 4050",
    "RTX 4060",
    "RTX 4070",
]

PREFERENCE_SCHEMA = {
    "version": "v1",
    "state_fields": STATE_FIELDS,
    "hard_fields": HARD_FIELDS,
    "semantic_fields": SEMANTIC_FIELDS,
    "canonical_brands": CANONICAL_BRANDS,
    "canonical_gpu_values": GPU_VALUES,
    "usage_values": USAGE_VALUES,
    "notes": {
        "budget_max": "Maximum acceptable product price.",
        "ram_min_gb": "Minimum acceptable RAM in GB.",
        "storage_min_gb": "Minimum acceptable storage in GB.",
        "screen_size_max_in": "Maximum acceptable screen size in inches.",
        "weight_max_kg": "Maximum acceptable weight in kilograms.",
        "usage": "Contextual intent only; not a deterministic hard filter.",
        "preference_notes": "Free-text semantic preferences; not a deterministic hard filter.",
    },
}

with open(SCHEMA_PATH, "w", encoding="utf-8") as f:
    json.dump(PREFERENCE_SCHEMA, f, ensure_ascii=False, indent=2)

print("Saved schema:", SCHEMA_PATH)


Saved schema: /content/drive/MyDrive/UTS Courses/Industrial Project/data/evaluation/preference_state_schema_v1.json


## 4. Oracle update/state functions

In [6]:

def blank_state():
    return copy.deepcopy(EMPTY_STATE)


def set_update(field, value):
    return {
        "field": field,
        "operation": "set",
        "value": value,
    }


def remove_update(field):
    return {
        "field": field,
        "operation": "remove",
        "value": None,
    }


def apply_oracle_updates(state, updates):
    new_state = copy.deepcopy(state)

    for update in updates:
        field = update["field"]
        operation = update["operation"]

        if field not in STATE_FIELDS:
            raise ValueError(f"Unknown state field: {field}")

        if operation == "set":
            new_state[field] = update["value"]
        elif operation == "remove":
            new_state[field] = None
        else:
            raise ValueError(f"Unsupported operation: {operation}")

    return new_state


def make_turn(
    user_text,
    scenario_type,
    updates,
    state_before,
    clarification_needed=False,
    clarification_question=None,
    notes=None,
):
    state_after = apply_oracle_updates(state_before, updates)

    return {
        "user_text": user_text,
        "scenario_type": scenario_type,
        "oracle_update": copy.deepcopy(updates),
        "oracle_state_before": copy.deepcopy(state_before),
        "oracle_state_after": copy.deepcopy(state_after),
        "oracle_clarification_needed": bool(clarification_needed),
        "oracle_clarification_question": clarification_question,
        "annotation_notes": notes,
    }, state_after



## 5. Deterministic retrieval oracle

This is **not** an experimental learned-retrieval metric.

It is used only to annotate each oracle state with the number of products that satisfy all active hard constraints. Missing product values do not satisfy an active constraint.


In [7]:

def count_candidates(state, df):
    mask = pd.Series(True, index=df.index)

    if state["budget_max"] is not None:
        mask &= df["price"].notna() & (df["price"] <= state["budget_max"])

    if state["brand"] is not None:
        mask &= df["brand"].notna() & (df["brand"] == state["brand"])

    if state["ram_min_gb"] is not None:
        mask &= df["ram_gb"].notna() & (df["ram_gb"] >= state["ram_min_gb"])

    if state["storage_min_gb"] is not None:
        mask &= (
            df["storage_gb"].notna()
            & (df["storage_gb"] >= state["storage_min_gb"])
        )

    if state["screen_size_max_in"] is not None:
        mask &= (
            df["screen_in"].notna()
            & (df["screen_in"] <= state["screen_size_max_in"])
        )

    if state["gpu_required"] is not None:
        mask &= df["gpu"].notna() & (df["gpu"] == state["gpu_required"])

    if state["weight_max_kg"] is not None:
        mask &= (
            df["weight_kg"].notna()
            & (df["weight_kg"] <= state["weight_max_kg"])
        )

    return int(mask.sum())


## 6. Split-specific natural-language templates

In [8]:

DEV_STYLE = {
    "budget": [
        "Keep it under ${v}.",
        "My budget is ${v} max.",
        "I don't want to spend more than ${v}.",
    ],
    "brand": [
        "I'd prefer {v}.",
        "Let's go with {v}.",
        "I want a {v} laptop.",
    ],
    "ram": [
        "I need at least {v}GB of RAM.",
        "{v}GB RAM minimum.",
        "Make the memory at least {v} gigs.",
    ],
    "storage": [
        "I need at least {v}GB of storage.",
        "Storage should be {v}GB or more.",
        "Give me at least {v} gigs of storage.",
    ],
    "screen": [
        "I want a screen no bigger than {v} inches.",
        "Keep the screen at {v} inches or smaller.",
        "Maximum screen size is {v} inches.",
    ],
    "gpu": [
        "I need {v} graphics.",
        "For graphics, require {v}.",
        "Make sure it has {v}.",
    ],
    "weight": [
        "Keep the weight under {v} kg.",
        "I want something no heavier than {v} kilograms.",
        "Maximum weight is {v} kg.",
    ],
    "usage": [
        "I'll mainly use it for {v}.",
        "This is mostly for {v}.",
        "My main use is {v}.",
    ],
}

TEST_STYLE = {
    "budget": [
        "I'd like to cap the spend at ${v}.",
        "Anything above ${v} is too much for me.",
        "My ceiling is around ${v}, don't go over it.",
        "Can we stay within ${v}?",
    ],
    "brand": [
        "I'm leaning toward {v}.",
        "{v} would be my preferred maker.",
        "I'd rather stick with {v}.",
        "Make the brand {v}.",
    ],
    "ram": [
        "I'd be comfortable with no less than {v}GB memory.",
        "Please make {v} gigs of RAM the floor.",
        "I don't want anything below {v}GB RAM.",
        "At least {v} gigs of memory, please.",
    ],
    "storage": [
        "I'd like {v}GB of storage at minimum.",
        "Don't show me anything with less than {v}GB storage.",
        "Set the storage floor to {v}GB.",
        "I need {v} gigs or more for storage.",
    ],
    "screen": [
        "I'd prefer the display to stay at or below {v} inches.",
        "Nothing larger than {v} inches.",
        "Cap the screen size at {v} inches.",
        "I want {v} inches or smaller.",
    ],
    "gpu": [
        "Graphics-wise, I specifically need {v}.",
        "The GPU requirement is {v}.",
        "I'd like the graphics to be {v}.",
        "Only keep machines matching {v} graphics.",
    ],
    "weight": [
        "Portability matters, so keep it to {v} kg or less.",
        "I don't want it heavier than {v} kilos.",
        "Set a {v} kg weight ceiling.",
        "Anything over {v} kg is too heavy.",
    ],
    "usage": [
        "The machine is mainly for {v}.",
        "Think of this as a {v} laptop for me.",
        "Most of what I'll do on it is {v}.",
        "My main workload is {v}.",
    ],
}


def choose_template(split, field_key, rng):
    bank = DEV_STYLE if split == "development" else TEST_STYLE
    return rng.choice(bank[field_key])


def realise_set(field, value, split, rng):
    key_map = {
        "budget_max": "budget",
        "brand": "brand",
        "ram_min_gb": "ram",
        "storage_min_gb": "storage",
        "screen_size_max_in": "screen",
        "gpu_required": "gpu",
        "weight_max_kg": "weight",
        "usage": "usage",
    }

    template = choose_template(split, key_map[field], rng)

    if field == "budget_max":
        return template.format(v=int(value))

    if field in {
        "ram_min_gb",
        "storage_min_gb",
    }:
        return template.format(v=int(value))

    if field in {
        "screen_size_max_in",
        "weight_max_kg",
    }:
        return template.format(v=value)

    return template.format(v=value)


DEV_REMOVE = [
    "Actually, forget the {label} requirement.",
    "I don't care about {label} anymore.",
    "Remove the {label} constraint.",
]

TEST_REMOVE = [
    "Drop the {label} condition now.",
    "That {label} restriction isn't important anymore.",
    "Take out the {label} requirement.",
    "Let's stop filtering on {label}.",
]

FIELD_LABELS = {
    "budget_max": "budget",
    "brand": "brand",
    "ram_min_gb": "RAM",
    "storage_min_gb": "storage",
    "screen_size_max_in": "screen size",
    "gpu_required": "GPU",
    "weight_max_kg": "weight",
    "usage": "usage",
    "preference_notes": "extra preference notes",
}


def realise_remove(field, split, rng):
    bank = DEV_REMOVE if split == "development" else TEST_REMOVE
    return rng.choice(bank).format(label=FIELD_LABELS[field])


## 7. Normalisation / semantic phrases

In [9]:
BRAND_ALIAS = {
    "HP": ["Hewlett Packard", "HP", "Pavilion"],
    "Lenovo": ["Lenovo", "ThinkPad", "IdeaPad"],
    "ASUS": ["ASUS", "ROG", "VivoBook"],
    "Dell": ["Dell", "Latitude", "Inspiron"],
    "Apple": ["Apple", "MacBook"],
    "Microsoft": ["Microsoft", "Surface"],
    "Acer": ["Acer", "Aspire"],
}

GPU_ALIAS = {
    "Integrated": [
        "integrated graphics",
        "an integrated GPU",
    ],
    "Intel Iris Xe": [
        "Iris Xe graphics",
        "Intel Iris Xe",
    ],
    "AMD Radeon": [
        "AMD Radeon graphics",
        "Radeon graphics",
    ],
    "RTX 3050": [
        "an RTX 3050",
        "a 3050 GPU",
    ],
    "RTX 3060": [
        "an RTX 3060",
        "a 3060 graphics card",
    ],
    "RTX 4060": [
        "an RTX 4060",
        "a 4060 GPU",
    ],
    "RTX 4070": [
        "an RTX 4070",
        "4070 graphics",
    ],
}

# IMPORTANT:
# Keep each natural-language utterance explicitly paired with its intended
# oracle preference_notes value. Do not maintain separate positional lists.
SOFT_PREFERENCE_PAIRS_DEV = [
    (
        "I usually work in the library, so I'd prefer something quiet.",
        "Prefers a quiet laptop for library use.",
    ),
    (
        "I take it between classes a lot, so portability matters to me.",
        "Often carries the laptop between classes and values portability.",
    ),
    (
        "I sometimes edit short videos as well.",
        "Occasionally edits short videos.",
    ),
    (
        "I'd rather not have a flashy gaming-looking machine.",
        "Prefers a simple laptop without flashy gaming styling.",
    ),
]

SOFT_PREFERENCE_PAIRS_TEST = [
    (
        "I'll be using it in quiet study spaces quite often, so low noise would be nice.",
        "Prefers a quiet laptop for library use.",
    ),
    (
        "It'll travel around campus with me most days, so I'd value something easy to carry.",
        "Often carries the laptop between classes and values portability.",
    ),
    (
        "Every now and then I cut together short videos, although that's not the main job.",
        "Occasionally edits short videos.",
    ),
    (
        "I'd prefer something understated rather than a laptop that looks aggressively gaming-focused.",
        "Prefers a simple laptop without flashy gaming styling.",
    ),
]


def make_soft_note(split, rng):
    """
    Return a semantically aligned (user_utterance, oracle_preference_note) pair.
    """
    bank = (
        SOFT_PREFERENCE_PAIRS_DEV
        if split == "development"
        else SOFT_PREFERENCE_PAIRS_TEST
    )
    return rng.choice(bank)

## 8. Dialogue recipe builders

In [10]:

def pick_different(rng, values, current=None):
    choices = [x for x in values if x != current]
    return rng.choice(choices)


def add_turn(dialogue, state, **kwargs):
    turn, new_state = make_turn(state_before=state, **kwargs)
    dialogue.append(turn)
    return new_state


def recipe_basic_accumulation(split, rng):
    dialogue, state = [], blank_state()

    budget = rng.choice(BUDGET_VALUES)
    ram = rng.choice([8, 16, 32])
    storage = rng.choice([256, 512, 1024])
    screen = rng.choice([14.0, 15.0, 16.0])

    for field, value, scenario in [
        ("budget_max", budget, "simple_set"),
        ("ram_min_gb", ram, "constraint_accumulation"),
        ("storage_min_gb", storage, "constraint_accumulation"),
        ("screen_size_max_in", screen, "constraint_accumulation"),
    ]:
        state = add_turn(
            dialogue,
            state,
            user_text=realise_set(field, value, split, rng),
            scenario_type=scenario,
            updates=[set_update(field, value)],
        )

    return dialogue


def recipe_multi_attribute(split, rng):
    dialogue, state = [], blank_state()

    budget = rng.choice(BUDGET_VALUES)
    ram = rng.choice([8, 16, 32])
    storage = rng.choice([256, 512, 1024])

    if split == "development":
        text = (
            f"I want at least {ram}GB RAM and {storage}GB storage, "
            f"and keep it under ${budget}."
        )
    else:
        text = (
            f"Let's make the minimum memory {ram} gigs, storage {storage}GB or more, "
            f"with a spending ceiling of ${budget}."
        )

    state = add_turn(
        dialogue,
        state,
        user_text=text,
        scenario_type="multi_attribute_set",
        updates=[
            set_update("ram_min_gb", ram),
            set_update("storage_min_gb", storage),
            set_update("budget_max", budget),
        ],
    )

    brand = rng.choice(CANONICAL_BRANDS)
    state = add_turn(
        dialogue,
        state,
        user_text=realise_set("brand", brand, split, rng),
        scenario_type="constraint_accumulation",
        updates=[set_update("brand", brand)],
    )

    return dialogue


def recipe_correction_removal(split, rng):
    dialogue, state = [], blank_state()

    initial_ram = rng.choice([8, 16])
    corrected_ram = 32 if initial_ram == 16 else 16

    state = add_turn(
        dialogue,
        state,
        user_text=realise_set("ram_min_gb", initial_ram, split, rng),
        scenario_type="simple_set",
        updates=[set_update("ram_min_gb", initial_ram)],
    )

    correction_text = (
        f"Actually, make that {corrected_ram}GB RAM instead."
        if split == "development"
        else f"Change that memory requirement — I want at least {corrected_ram}GB now."
    )

    state = add_turn(
        dialogue,
        state,
        user_text=correction_text,
        scenario_type="correction_overwrite",
        updates=[set_update("ram_min_gb", corrected_ram)],
    )

    brand = rng.choice(["HP", "Lenovo", "Dell", "ASUS"])
    state = add_turn(
        dialogue,
        state,
        user_text=realise_set("brand", brand, split, rng),
        scenario_type="simple_set",
        updates=[set_update("brand", brand)],
    )

    state = add_turn(
        dialogue,
        state,
        user_text=realise_remove("brand", split, rng),
        scenario_type="removal",
        updates=[remove_update("brand")],
    )

    return dialogue


def recipe_cross_turn_reference(split, rng):
    dialogue, state = [], blank_state()

    first_brand = rng.choice(["HP", "Lenovo", "Dell", "ASUS"])
    second_brand = pick_different(
        rng,
        ["HP", "Lenovo", "Dell", "ASUS"],
        first_brand,
    )

    state = add_turn(
        dialogue,
        state,
        user_text=realise_set("brand", first_brand, split, rng),
        scenario_type="simple_set",
        updates=[set_update("brand", first_brand)],
    )

    text = (
        f"Actually make that {second_brand} instead."
        if split == "development"
        else f"On second thought, switch that manufacturer to {second_brand}."
    )

    state = add_turn(
        dialogue,
        state,
        user_text=text,
        scenario_type="cross_turn_reference",
        updates=[set_update("brand", second_brand)],
    )

    state = add_turn(
        dialogue,
        state,
        user_text=realise_set("storage_min_gb", 512, split, rng),
        scenario_type="constraint_accumulation",
        updates=[set_update("storage_min_gb", 512)],
    )

    return dialogue


def recipe_units_numeric(split, rng):
    dialogue, state = [], blank_state()

    budget_text = (
        "My budget is one and a half grand."
        if split == "development"
        else "Try to keep the total spend below 1.5k."
    )
    state = add_turn(
        dialogue,
        state,
        user_text=budget_text,
        scenario_type="numeric_normalization",
        updates=[set_update("budget_max", 1500)],
    )

    storage_text = (
        "I need at least one terabyte of storage."
        if split == "development"
        else "Storage-wise, I want a full 1 TB minimum."
    )
    state = add_turn(
        dialogue,
        state,
        user_text=storage_text,
        scenario_type="unit_conversion",
        updates=[set_update("storage_min_gb", 1024)],
    )

    weight_text = (
        "Keep it under 1500 grams."
        if split == "development"
        else "I'd like the machine to weigh no more than 1,500 g."
    )
    state = add_turn(
        dialogue,
        state,
        user_text=weight_text,
        scenario_type="unit_conversion",
        updates=[set_update("weight_max_kg", 1.5)],
    )

    return dialogue


def recipe_brand_gpu_normalization(split, rng):
    dialogue, state = [], blank_state()

    canonical_brand = rng.choice(list(BRAND_ALIAS.keys()))
    alias = rng.choice(BRAND_ALIAS[canonical_brand])

    if split == "development":
        brand_text = f"I'd like a {alias}."
    else:
        brand_text = f"For the manufacturer, I'm thinking {alias}."

    state = add_turn(
        dialogue,
        state,
        user_text=brand_text,
        scenario_type="brand_normalization",
        updates=[set_update("brand", canonical_brand)],
    )

    canonical_gpu = rng.choice(list(GPU_ALIAS.keys()))
    alias_gpu = rng.choice(GPU_ALIAS[canonical_gpu])

    if split == "development":
        gpu_text = f"I also need {alias_gpu}."
    else:
        gpu_text = f"Graphics are important too — require {alias_gpu}."

    state = add_turn(
        dialogue,
        state,
        user_text=gpu_text,
        scenario_type="gpu_normalization",
        updates=[set_update("gpu_required", canonical_gpu)],
    )

    return dialogue


def recipe_semantic_preferences(split, rng):
    dialogue, state = [], blank_state()

    usage = rng.choice(USAGE_VALUES)

    state = add_turn(
        dialogue,
        state,
        user_text=realise_set("usage", usage, split, rng),
        scenario_type="semantic_usage",
        updates=[set_update("usage", usage)],
    )

    utterance, note = make_soft_note(split, rng)
    state = add_turn(
        dialogue,
        state,
        user_text=utterance,
        scenario_type="soft_preference",
        updates=[set_update("preference_notes", note)],
    )

    return dialogue


def recipe_mixed_hard_soft(split, rng):
    dialogue, state = [], blank_state()

    ram = rng.choice([8, 16, 32])
    budget = rng.choice([1000, 1200, 1500, 1800])
    usage = rng.choice(USAGE_VALUES)
    utterance, note = make_soft_note(split, rng)

    if split == "development":
        text = (
            f"I need at least {ram}GB RAM under ${budget}. "
            f"I'll mainly use it for {usage}. {utterance}"
        )
    else:
        text = (
            f"Keep the budget at ${budget} or below and memory at {ram}GB minimum. "
            f"It's mainly for {usage}. {utterance}"
        )

    state = add_turn(
        dialogue,
        state,
        user_text=text,
        scenario_type="mixed_hard_soft",
        updates=[
            set_update("ram_min_gb", ram),
            set_update("budget_max", budget),
            set_update("usage", usage),
            set_update("preference_notes", note),
        ],
    )

    return dialogue


def recipe_ambiguity_resolution(split, rng):
    dialogue, state = [], blank_state()

    vague = (
        "I want something pretty light."
        if split == "development"
        else "I'd like it to be on the lighter side if possible."
    )

    state = add_turn(
        dialogue,
        state,
        user_text=vague,
        scenario_type="ambiguity_clarification",
        updates=[],
        clarification_needed=True,
        clarification_question="What maximum laptop weight would you like?",
        notes="Qualitative lightweight preference is insufficient for a numeric hard constraint.",
    )

    weight = rng.choice([1.5, 1.8, 2.0])

    resolved = (
        f"Let's say no more than {weight} kg."
        if split == "development"
        else f"Okay, set the upper limit to {weight} kilograms."
    )

    state = add_turn(
        dialogue,
        state,
        user_text=resolved,
        scenario_type="ambiguity_resolution",
        updates=[set_update("weight_max_kg", weight)],
    )

    return dialogue


def recipe_contradictory_same_turn(split, rng):
    dialogue, state = [], blank_state()

    text = (
        "I want a 14-inch laptop, but it also has to be at least 17 inches."
        if split == "development"
        else "Keep the display to 14 inches max, but I also need 17 inches or larger."
    )

    state = add_turn(
        dialogue,
        state,
        user_text=text,
        scenario_type="contradictory_same_turn",
        updates=[],
        clarification_needed=True,
        clarification_question="Which screen-size requirement should I follow?",
    )

    resolved_screen = rng.choice([14.0, 17.0])

    resolution = (
        f"Go with {resolved_screen} inches maximum."
        if split == "development"
        else f"Use {resolved_screen} inches as the maximum size."
    )

    state = add_turn(
        dialogue,
        state,
        user_text=resolution,
        scenario_type="contradiction_resolution",
        updates=[set_update("screen_size_max_in", resolved_screen)],
    )

    return dialogue


def recipe_noop_smalltalk(split, rng):
    dialogue, state = [], blank_state()

    state = add_turn(
        dialogue,
        state,
        user_text=realise_set("budget_max", 1200, split, rng),
        scenario_type="simple_set",
        updates=[set_update("budget_max", 1200)],
    )

    noop_text = (
        "Thanks, that makes sense."
        if split == "development"
        else "Okay, sounds good so far."
    )

    state = add_turn(
        dialogue,
        state,
        user_text=noop_text,
        scenario_type="no_op",
        updates=[],
    )

    return dialogue


def recipe_complex(split, rng):
    dialogue, state = [], blank_state()

    brand = rng.choice(["HP", "Lenovo", "Dell", "ASUS", "Acer"])
    state = add_turn(
        dialogue,
        state,
        user_text=realise_set("brand", brand, split, rng),
        scenario_type="simple_set",
        updates=[set_update("brand", brand)],
    )

    ram = rng.choice([16, 32])
    storage = rng.choice([512, 1024])

    if split == "development":
        multi = f"Add at least {ram}GB RAM and {storage}GB storage."
    else:
        multi = f"Memory should be {ram}GB minimum, and storage no less than {storage}GB."

    state = add_turn(
        dialogue,
        state,
        user_text=multi,
        scenario_type="multi_attribute_set",
        updates=[
            set_update("ram_min_gb", ram),
            set_update("storage_min_gb", storage),
        ],
    )

    utterance, note = make_soft_note(split, rng)
    state = add_turn(
        dialogue,
        state,
        user_text=utterance,
        scenario_type="soft_preference",
        updates=[set_update("preference_notes", note)],
    )

    state = add_turn(
        dialogue,
        state,
        user_text=realise_remove("brand", split, rng),
        scenario_type="removal",
        updates=[remove_update("brand")],
    )

    return dialogue


RECIPES = [
    ("basic_accumulation", recipe_basic_accumulation),
    ("multi_attribute", recipe_multi_attribute),
    ("correction_removal", recipe_correction_removal),
    ("cross_turn_reference", recipe_cross_turn_reference),
    ("units_numeric", recipe_units_numeric),
    ("brand_gpu_normalization", recipe_brand_gpu_normalization),
    ("semantic_preferences", recipe_semantic_preferences),
    ("mixed_hard_soft", recipe_mixed_hard_soft),
    ("ambiguity_resolution", recipe_ambiguity_resolution),
    ("contradictory_same_turn", recipe_contradictory_same_turn),
    ("noop_smalltalk", recipe_noop_smalltalk),
    ("complex", recipe_complex),
]


## 9. Generate development and held-out test dialogues

In [11]:

def generate_dialogues(split, n_dialogues, seed):
    rng = random.Random(seed)
    dialogues = []

    for i in range(n_dialogues):
        recipe_name, recipe_fn = RECIPES[i % len(RECIPES)]

        # Each recipe gets the same split-specific RNG stream but produces
        # different values and utterance templates.
        turns = recipe_fn(split, rng)

        for turn_index, turn in enumerate(turns, start=1):
            turn["turn_index"] = turn_index
            turn["oracle_candidate_count"] = count_candidates(
                turn["oracle_state_after"],
                products,
            )

        dialogues.append({
            "dialogue_id": f"{'dev' if split == 'development' else 'test'}_{i+1:04d}",
            "split": split,
            "source": "controlled_synthetic",
            "recipe": recipe_name,
            "turns": turns,
        })

    return dialogues


dev_dialogues = generate_dialogues(
    "development",
    N_DEV_DIALOGUES,
    DEV_SEED,
)

test_dialogues = generate_dialogues(
    "heldout_test",
    N_TEST_GENERATED_DIALOGUES,
    TEST_SEED,
)

print("Development dialogues:", len(dev_dialogues))
print("Generated held-out test dialogues:", len(test_dialogues))


Development dialogues: 60
Generated held-out test dialogues: 120



## 10. Add a small manually authored adversarial held-out subset

These dialogues are deliberately written separately from the template generator.

They remain part of the held-out test split.


In [12]:

def build_manual_dialogue(dialogue_id, turn_specs):
    state = blank_state()
    turns = []

    for i, spec in enumerate(turn_specs, start=1):
        turn, state = make_turn(
            user_text=spec["user_text"],
            scenario_type=spec["scenario_type"],
            updates=spec.get("updates", []),
            state_before=state,
            clarification_needed=spec.get("clarification_needed", False),
            clarification_question=spec.get("clarification_question"),
            notes=spec.get("notes"),
        )
        turn["turn_index"] = i
        turn["oracle_candidate_count"] = count_candidates(
            turn["oracle_state_after"],
            products,
        )
        turns.append(turn)

    return {
        "dialogue_id": dialogue_id,
        "split": "heldout_test",
        "source": "manual_adversarial",
        "recipe": "manual_adversarial",
        "turns": turns,
    }


manual_specs = [
    [
        {
            "user_text": "I'm mostly doing uni work, but I also edit a short video every couple of weeks. Keep it under fifteen hundred and give me at least sixteen gigs of memory.",
            "scenario_type": "mixed_hard_soft",
            "updates": [
                set_update("usage", "study"),
                set_update("preference_notes", "Occasionally edits short videos."),
                set_update("budget_max", 1500),
                set_update("ram_min_gb", 16),
            ],
        },
        {
            "user_text": "Actually, forget the budget limit. I care more about keeping it under 1.5 kilos.",
            "scenario_type": "removal_and_set",
            "updates": [
                remove_update("budget_max"),
                set_update("weight_max_kg", 1.5),
            ],
        },
    ],
    [
        {
            "user_text": "A ThinkPad would be ideal, 512 gigs minimum.",
            "scenario_type": "brand_normalization",
            "updates": [
                set_update("brand", "Lenovo"),
                set_update("storage_min_gb", 512),
            ],
        },
        {
            "user_text": "Make that one terabyte actually.",
            "scenario_type": "cross_turn_reference",
            "updates": [
                set_update("storage_min_gb", 1024),
            ],
        },
    ],
    [
        {
            "user_text": "I don't need anything huge. Fourteen inches or below, and something light would be great.",
            "scenario_type": "mixed_hard_ambiguous",
            "updates": [
                set_update("screen_size_max_in", 14.0),
            ],
            "clarification_needed": True,
            "clarification_question": "What maximum laptop weight would you like?",
        },
        {
            "user_text": "Call the weight limit 1.8 kg.",
            "scenario_type": "ambiguity_resolution",
            "updates": [
                set_update("weight_max_kg", 1.8),
            ],
        },
    ],
    [
        {
            "user_text": "I'd like a MacBook, but don't make Apple a hard requirement if that narrows things too much.",
            "scenario_type": "semantic_brand_preference",
            "updates": [
                set_update("preference_notes", "Prefers Apple/MacBook but does not want brand enforced as a hard constraint."),
            ],
        },
    ],
    [
        {
            "user_text": "I need a 4060 for gaming, 16GB RAM minimum, and no more than two grand.",
            "scenario_type": "multi_attribute_set",
            "updates": [
                set_update("gpu_required", "RTX 4060"),
                set_update("ram_min_gb", 16),
                set_update("budget_max", 2000),
                set_update("usage", "gaming"),
            ],
        },
        {
            "user_text": "Actually the 4060 isn't mandatory anymore.",
            "scenario_type": "removal",
            "updates": [
                remove_update("gpu_required"),
            ],
        },
    ],
    [
        {
            "user_text": "I want something under $900 and at least 32GB RAM.",
            "scenario_type": "multi_attribute_set",
            "updates": [
                set_update("budget_max", 900),
                set_update("ram_min_gb", 32),
            ],
        },
        {
            "user_text": "That's probably too restrictive. Keep the RAM but raise the budget ceiling to $1400.",
            "scenario_type": "correction_overwrite",
            "updates": [
                set_update("budget_max", 1400),
            ],
        },
    ],
    [
        {
            "user_text": "A Surface would be nice, and I need 256GB or more.",
            "scenario_type": "brand_normalization",
            "updates": [
                set_update("brand", "Microsoft"),
                set_update("storage_min_gb", 256),
            ],
        },
        {
            "user_text": "Forget the manufacturer part.",
            "scenario_type": "cross_turn_removal",
            "updates": [
                remove_update("brand"),
            ],
        },
    ],
    [
        {
            "user_text": "I need something for business, professional looking, not flashy. Budget is $1800.",
            "scenario_type": "mixed_hard_soft",
            "updates": [
                set_update("usage", "business"),
                set_update("preference_notes", "Prefers a professional-looking laptop without flashy styling."),
                set_update("budget_max", 1800),
            ],
        },
    ],
    [
        {
            "user_text": "Give me 16GB RAM and 512GB storage. Actually, make the RAM 32.",
            "scenario_type": "contradictory_resolved_same_turn",
            "updates": [
                set_update("ram_min_gb", 32),
                set_update("storage_min_gb", 512),
            ],
        },
    ],
    [
        {
            "user_text": "I want it to be both under 13 inches and at least 17 inches.",
            "scenario_type": "contradictory_same_turn",
            "updates": [],
            "clarification_needed": True,
            "clarification_question": "Which screen-size requirement should I follow?",
        },
        {
            "user_text": "Use 13 inches as the maximum.",
            "scenario_type": "contradiction_resolution",
            "updates": [
                set_update("screen_size_max_in", 13.0),
            ],
        },
    ],
]

manual_dialogues = [
    build_manual_dialogue(
        f"test_manual_{i+1:03d}",
        spec,
    )
    for i, spec in enumerate(manual_specs)
]

test_dialogues.extend(manual_dialogues)

print("Manual adversarial dialogues:", len(manual_dialogues))
print("Total held-out test dialogues:", len(test_dialogues))


Manual adversarial dialogues: 10
Total held-out test dialogues: 130


## 11. Validation checks

In [13]:

def validate_dialogues(dialogues):
    errors = []

    dialogue_ids = [d["dialogue_id"] for d in dialogues]
    if len(dialogue_ids) != len(set(dialogue_ids)):
        errors.append("Duplicate dialogue_id detected.")

    for dialogue in dialogues:
        previous_state = blank_state()

        for turn in dialogue["turns"]:
            if set(turn["oracle_state_before"].keys()) != set(STATE_FIELDS):
                errors.append(
                    f"{dialogue['dialogue_id']} has invalid state-before keys."
                )

            if set(turn["oracle_state_after"].keys()) != set(STATE_FIELDS):
                errors.append(
                    f"{dialogue['dialogue_id']} has invalid state-after keys."
                )

            if turn["oracle_state_before"] != previous_state:
                errors.append(
                    f"{dialogue['dialogue_id']} turn {turn['turn_index']} "
                    "state continuity error."
                )

            recomputed = apply_oracle_updates(
                turn["oracle_state_before"],
                turn["oracle_update"],
            )

            if recomputed != turn["oracle_state_after"]:
                errors.append(
                    f"{dialogue['dialogue_id']} turn {turn['turn_index']} "
                    "oracle update/state mismatch."
                )

            previous_state = copy.deepcopy(turn["oracle_state_after"])

    return errors


dev_errors = validate_dialogues(dev_dialogues)
test_errors = validate_dialogues(test_dialogues)

dev_texts = {
    turn["user_text"].strip().lower()
    for d in dev_dialogues
    for turn in d["turns"]
}
test_texts = {
    turn["user_text"].strip().lower()
    for d in test_dialogues
    for turn in d["turns"]
}

exact_text_overlap = dev_texts.intersection(test_texts)

print("Development validation errors:", len(dev_errors))
print("Test validation errors:", len(test_errors))
print("Exact dev/test utterance overlap:", len(exact_text_overlap))

assert not dev_errors, dev_errors[:10]
assert not test_errors, test_errors[:10]

# Exact overlap should normally be zero because template banks are split-specific.
if exact_text_overlap:
    print("Warning: exact utterance overlap examples:")
    print(list(exact_text_overlap)[:10])
else:
    print("No exact user-utterance overlap between development and held-out test.")


Development validation errors: 0
Test validation errors: 0
Exact dev/test utterance overlap: 0
No exact user-utterance overlap between development and held-out test.



## 11.1 Semantic preference oracle-label sanity check

This specifically verifies the bug fixed in this version: every generated
soft-preference utterance must map to its intended `preference_notes` oracle value.


In [14]:

EXPECTED_SOFT_NOTE_MAP = dict(
    SOFT_PREFERENCE_PAIRS_DEV + SOFT_PREFERENCE_PAIRS_TEST
)

semantic_mapping_errors = []

for dialogue in dev_dialogues + test_dialogues:
    for turn in dialogue["turns"]:
        user_text = turn["user_text"]

        if user_text not in EXPECTED_SOFT_NOTE_MAP:
            continue

        expected_note = EXPECTED_SOFT_NOTE_MAP[user_text]

        note_updates = [
            u for u in turn["oracle_update"]
            if u["field"] == "preference_notes"
            and u["operation"] == "set"
        ]

        if len(note_updates) != 1:
            semantic_mapping_errors.append({
                "dialogue_id": dialogue["dialogue_id"],
                "turn_index": turn["turn_index"],
                "user_text": user_text,
                "problem": f"Expected exactly one preference_notes update, found {len(note_updates)}",
            })
            continue

        actual_note = note_updates[0]["value"]

        if actual_note != expected_note:
            semantic_mapping_errors.append({
                "dialogue_id": dialogue["dialogue_id"],
                "turn_index": turn["turn_index"],
                "user_text": user_text,
                "expected_note": expected_note,
                "actual_note": actual_note,
            })

print("Semantic preference mapping errors:", len(semantic_mapping_errors))

if semantic_mapping_errors:
    display(pd.DataFrame(semantic_mapping_errors))
else:
    print("PASS — all generated soft-preference utterances match their oracle preference_notes labels.")

assert not semantic_mapping_errors


Semantic preference mapping errors: 0
PASS — all generated soft-preference utterances match their oracle preference_notes labels.


## 12. Flatten turns for easy inspection / later evaluation

In [15]:

def flatten_dialogues(dialogues):
    rows = []

    for dialogue in dialogues:
        for turn in dialogue["turns"]:
            rows.append({
                "dialogue_id": dialogue["dialogue_id"],
                "split": dialogue["split"],
                "source": dialogue["source"],
                "recipe": dialogue["recipe"],
                "turn_index": turn["turn_index"],
                "scenario_type": turn["scenario_type"],
                "user_text": turn["user_text"],
                "oracle_update_json": json.dumps(
                    turn["oracle_update"],
                    ensure_ascii=False,
                    sort_keys=True,
                ),
                "oracle_state_before_json": json.dumps(
                    turn["oracle_state_before"],
                    ensure_ascii=False,
                    sort_keys=True,
                ),
                "oracle_state_after_json": json.dumps(
                    turn["oracle_state_after"],
                    ensure_ascii=False,
                    sort_keys=True,
                ),
                "oracle_clarification_needed": turn[
                    "oracle_clarification_needed"
                ],
                "oracle_clarification_question": turn[
                    "oracle_clarification_question"
                ],
                "oracle_candidate_count": turn[
                    "oracle_candidate_count"
                ],
                "annotation_notes": turn["annotation_notes"],
            })

    return pd.DataFrame(rows)


dev_turns = flatten_dialogues(dev_dialogues)
test_turns = flatten_dialogues(test_dialogues)

print("Development turns:", len(dev_turns))
print("Held-out test turns:", len(test_turns))

display(dev_turns.head(20))


Development turns: 155
Held-out test turns: 327


,dialogue_id,split,source,recipe,turn_index,scenario_type,user_text,oracle_update_json,oracle_state_before_json,oracle_state_after_json,oracle_clarification_needed,oracle_clarification_question,oracle_candidate_count,annotation_notes
0,dev_0001,development,controlled_synthetic,basic_accumulation,1,simple_set,Keep it under $2000.,"[{""field"": ""budget_max"", ""operation"": ""set"", ""value"": 2000}]","{""brand"": null, ""budget_max"": null, ""gpu_required"": null, ""preference_notes"": null, ""ram_min_gb"": null, ""screen_size_max_in"": null, ""storage_min_gb"": null, ""usage"": null, ""weight_max_kg"": null}","{""brand"": null, ""budget_max"": 2000, ""gpu_required"": null, ""preference_notes"": null, ""ram_min_gb"": null, ""screen_size_max_in"": null, ""storage_min_gb"": null, ""usage"": null, ""weight_max_kg"": null}",False,None,4881,None
1,dev_0001,development,controlled_synthetic,basic_accumulation,2,constraint_accumulation,32GB RAM minimum.,"[{""field"": ""ram_min_gb"", ""operation"": ""set"", ""value"": 32}]","{""brand"": null, ""budget_max"": 2000, ""gpu_required"": null, ""preference_notes"": null, ""ram_min_gb"": null, ""screen_size_max_in"": null, ""storage_min_gb"": null, ""usage"": null, ""weight_max_kg"": null}","{""brand"": null, ""budget_max"": 2000, ""gpu_required"": null, ""preference_notes"": null, ""ram_min_gb"": 32, ""screen_size_max_in"": null, ""storage_min_gb"": null, ""usage"": null, ""weight_max_kg"": null}",False,None,123,None
2,dev_0001,development,controlled_synthetic,basic_accumulation,3,constraint_accumulation,I need at least 1024GB of storage.,"[{""field"": ""storage_min_gb"", ""operation"": ""set"", ""value"": 1024}]","{""brand"": null, ""budget_max"": 2000, ""gpu_required"": null, ""preference_notes"": null, ""ram_min_gb"": 32, ""screen_size_max_in"": null, ""storage_min_gb"": null, ""usage"": null, ""weight_max_kg"": null}","{""brand"": null, ""budget_max"": 2000, ""gpu_required"": null, ""preference_notes"": null, ""ram_min_gb"": 32, ""screen_size_max_in"": null, ""storage_min_gb"": 1024, ""usage"": null, ""weight_max_kg"": null}",False,None,15,None
3,dev_0001,development,controlled_synthetic,basic_accumulation,4,constraint_accumulation,Maximum screen size is 14.0 inches.,"[{""field"": ""screen_size_max_in"", ""operation"": ""set"", ""value"": 14.0}]","{""brand"": null, ""budget_max"": 2000, ""gpu_required"": null, ""preference_notes"": null, ""ram_min_gb"": 32, ""screen_size_max_in"": null, ""storage_min_gb"": 1024, ""usage"": null, ""weight_max_kg"": null}","{""brand"": null, ""budget_max"": 2000, ""gpu_required"": null, ""preference_notes"": null, ""ram_min_gb"": 32, ""screen_size_max_in"": 14.0, ""storage_min_gb"": 1024, ""usage"": null, ""weight_max_kg"": null}",False,None,0,None
4,dev_0002,development,controlled_synthetic,multi_attribute,1,multi_attribute_set,"I want at least 32GB RAM and 256GB storage, and keep it under $1000.","[{""field"": ""ram_min_gb"", ""operation"": ""set"", ""value"": 32}, {""field"": ""storage_min_gb"", ""operation"": ""set"", ""value"": 256}, {""field"": ""budget_max"", ""operation"": ""set"", ""value"": 1000}]","{""brand"": null, ""budget_max"": null, ""gpu_required"": null, ""preference_notes"": null, ""ram_min_gb"": null, ""screen_size_max_in"": null, ""storage_min_gb"": null, ""usage"": null, ""weight_max_kg"": null}","{""brand"": null, ""budget_max"": 1000, ""gpu_required"": null, ""preference_notes"": null, ""ram_min_gb"": 32, ""screen_size_max_in"": null, ""storage_min_gb"": 256, ""usage"": null, ""weight_max_kg"": null}",False,None,64,None
5,dev_0002,development,controlled_synthetic,multi_attribute,2,constraint_accumulation,Let's go with LG.,"[{""field"": ""brand"", ""operation"": ""set"", ""value"": ""LG""}]","{""brand"": null, ""budget_max"": 1000, ""gpu_required"": null, ""preference_notes"": null, ""ram_min_gb"": 32, ""screen_size_max_in"": null, ""storage_min_gb"": 256, ""usage"": null, ""weight_max_kg"": null}","{""brand"": ""LG"", ""budget_max""

## 13. Scenario coverage and candidate-count sanity check

In [16]:

def scenario_summary(df):
    return (
        df["scenario_type"]
        .value_counts()
        .rename_axis("scenario_type")
        .reset_index(name="turns")
    )


print("DEVELOPMENT SCENARIOS")
display(scenario_summary(dev_turns))

print("\nHELD-OUT TEST SCENARIOS")
display(scenario_summary(test_turns))

print("\nCandidate-count summary — development")
display(dev_turns["oracle_candidate_count"].describe())

print("\nCandidate-count summary — held-out test")
display(test_turns["oracle_candidate_count"].describe())

print(
    "\nHeld-out turns with zero deterministic candidates:",
    int((test_turns["oracle_candidate_count"] == 0).sum()),
    "/",
    len(test_turns),
)


DEVELOPMENT SCENARIOS


,scenario_type,turns
0,simple_set,30
1,constraint_accumulation,25
2,multi_attribute_set,10
3,removal,10
4,unit_conversion,10
5,soft_preference,10
6,cross_turn_reference,5
7,numeric_normalization,5
8,brand_normalization,5
9,correction_overwrite,5



HELD-OUT TEST SCENARIOS


,scenario_type,turns
0,simple_set,60
1,constraint_accumulation,50
2,multi_attribute_set,22
3,removal,21
4,unit_conversion,20
5,soft_preference,20
6,mixed_hard_soft,12
7,brand_normalization,12
8,ambiguity_resolution,11
9,correction_overwrite,11



Candidate-count summary — development


,oracle_candidate_count
count,155.000000
mean,1825.406452
std,2002.812603
min,0.000000
25%,29.000000
50%,1226.000000
75%,4598.000000
max,4938.000000



Candidate-count summary — held-out test


,oracle_candidate_count
count,327.000000
mean,1831.140673
std,1965.156624
min,0.000000
25%,86.000000
50%,1019.000000
75%,4598.000000
max,4938.000000



Held-out turns with zero deterministic candidates: 11 / 327


## 14. Inspect random held-out examples

In [17]:

sample_cols = [
    "dialogue_id",
    "turn_index",
    "scenario_type",
    "user_text",
    "oracle_update_json",
    "oracle_state_after_json",
    "oracle_clarification_needed",
    "oracle_candidate_count",
]

display(
    test_turns.sample(
        min(30, len(test_turns)),
        random_state=42,
    )[sample_cols]
)


,dialogue_id,turn_index,scenario_type,user_text,oracle_update_json,oracle_state_after_json,oracle_clarification_needed,oracle_candidate_count
231,test_0089,2,unit_conversion,"Storage-wise, I want a full 1 TB minimum.","[{""field"": ""storage_min_gb"", ""operation"": ""set"", ""value"": 1024}]","{""brand"": null, ""budget_max"": 1500, ""gpu_required"": null, ""preference_notes"": null, ""ram_min_gb"": null, ""screen_size_max_in"": null, ""storage_min_gb"": 1024, ""usage"": null, ""weight_max_kg"": null}",False,29
110,test_0042,2,gpu_normalization,Graphics are important too — require an RTX 3050.,"[{""field"": ""gpu_required"", ""operation"": ""set"", ""value"": ""RTX 3050""}]","{""brand"": ""Microsoft"", ""budget_max"": null, ""gpu_required"": ""RTX 3050"", ""preference_notes"": null, ""ram_min_gb"": null, ""screen_size_max_in"": null, ""storage_min_gb"": null, ""usage"": null, ""weight_max_kg"": null}",False,0
250,test_0097,3,constraint_accumulation,I need 256 gigs or more for storage.,"[{""field"": ""storage_min_gb"", ""operation"": ""set"", ""value"": 256}]","{""brand"": null, ""budget_max"": 1200, ""gpu_required"": null, ""preference_notes"": null, ""ram_min_gb"": 16, ""screen_size_max_in"": null, ""storage_min_gb"": 256, ""usage"": null, ""weight_max_kg"": null}",False,1061
9,test_0003,4,removal,Drop the brand condition now.,"[{""field"": ""brand"", ""operation"": ""remove"", ""value"": null}]","{""brand"": null, ""budget_max"": null, ""gpu_required"": null, ""preference_notes"": null, ""ram_min_gb"": 32, ""screen_size_max_in"": null, ""storage_min_gb"": null, ""usage"": null, ""weight_max_kg"": null}",False,151
93,test_0037,1,simple_set,Can we stay within $1200?,"[{""field"": ""budget_max"", ""operation"": ""set"", ""value"": 1200}]","{""brand"": null, ""budget_max"": 1200, ""gpu_required"": null, ""preference_notes"": null, ""ram_min_gb"": null, ""screen_size_max_in"": null, ""storage_min_gb"": null, ""usage"": null, ""weight_max_kg"": null}",False,4598
220,test_0085,4,constraint_accumulation,Nothing larger than 16.0 inches.,"[{""field"": ""screen_size_max_in"", ""operation"": ""set"", ""value"": 16.0}]","{""brand"": null, ""budget_max"": 800, ""gpu_required"": null, ""preference_notes"": null, ""ram_min_gb"": 32, ""screen_size_max_in"": 16.0, ""storage_min_gb"": 1024, ""usage"": null, ""weight_max_kg"": null}",False,2
275,test_0108,1,simple_set,I'd rather stick with Dell.,"[{""field"": ""brand"", ""operation"": ""set"", ""value"": ""Dell""}]","{""brand"": ""Dell"", ""budget_max"": null, ""gpu_required"": null, ""preference_notes"": null, ""ram_min_gb"": null, ""screen_size_max_in"": null, ""storage_min_gb"": null, ""usage"": null, ""weight_max_kg"": null}",False,1019
199,test_0077,1,numeric_normalization,Try to keep the total spend below 1.5k.,"[{""field"": ""budget_max"", ""operation"": ""set"", ""value"": 1500}]","{""brand"": null, ""budget_max"": 1500, ""gpu_required"": null, ""preference_notes"": null, ""ram_min_gb"": null, ""screen_size_max_in"": null, ""storage_min_gb"": null, ""usage"": null, ""weight_max_kg"": null}",False,4768
204,test_0079,1,semantic_usage,Most of what I'll do on it is gaming.,"[{""field"": ""usage"", ""operation"": ""set"", ""value"": ""gaming""}]","{""brand"": null, ""budget_max"": null, ""gpu_required"": null, ""preference_notes"": null, ""ram_min_gb"": null, ""screen_size_max_in"": null, ""storage_min_gb"": null, ""usage"": ""gaming"", ""weight_max_kg"": null}",False,4938
101,test_0039,3,simple_set,I'd rather stick with Dell.,"[{""field"": ""brand"", ""operation"": ""set"", ""value"": ""Dell""}]","{""brand"": ""Dell"", ""budget_max"": null, ""gpu_required"": null, ""preference_notes"": null, ""ram_min_gb"": 32, ""screen_size_max_in"": null, ""storage_min_gb"": null, ""usage"": null, ""weight_max_kg"": null}",False,68


## 15. Export datasets

In [18]:

def write_jsonl(path, records):
    with open(path, "w", encoding="utf-8") as f:
        for record in records:
            f.write(
                json.dumps(
                    record,
                    ensure_ascii=False,
                    separators=(",", ":"),
                )
                + "\n"
            )


write_jsonl(DEV_DIALOGUES_PATH, dev_dialogues)
write_jsonl(TEST_DIALOGUES_PATH, test_dialogues)

dev_turns.to_csv(DEV_TURNS_PATH, index=False)
test_turns.to_csv(TEST_TURNS_PATH, index=False)

summary = {
    "version": "v1",
    "product_database_path": str(PRODUCT_DB_PATH),
    "product_database_rows": int(len(products)),
    "development": {
        "dialogues": int(len(dev_dialogues)),
        "turns": int(len(dev_turns)),
        "source_counts": {
            str(k): int(v)
            for k, v in Counter(
                d["source"] for d in dev_dialogues
            ).items()
        },
        "scenario_counts": {
            str(k): int(v)
            for k, v in dev_turns[
                "scenario_type"
            ].value_counts().items()
        },
    },
    "heldout_test": {
        "dialogues": int(len(test_dialogues)),
        "turns": int(len(test_turns)),
        "manual_adversarial_dialogues": int(len(manual_dialogues)),
        "source_counts": {
            str(k): int(v)
            for k, v in Counter(
                d["source"] for d in test_dialogues
            ).items()
        },
        "scenario_counts": {
            str(k): int(v)
            for k, v in test_turns[
                "scenario_type"
            ].value_counts().items()
        },
        "zero_candidate_turns": int(
            (test_turns["oracle_candidate_count"] == 0).sum()
        ),
    },
    "dev_test_exact_utterance_overlap": int(len(exact_text_overlap)),
    "methodology_note": (
        "No model training or fine-tuning is performed. "
        "The development set is used for prompt/schema/validation refinement. "
        "The held-out test set is reserved for final evaluation. "
        "Ground-truth semantic updates and states are defined before "
        "natural-language realization."
    ),
}

with open(SUMMARY_PATH, "w", encoding="utf-8") as f:
    json.dump(summary, f, ensure_ascii=False, indent=2)

print("Saved:")
print("-", DEV_DIALOGUES_PATH)
print("-", TEST_DIALOGUES_PATH)
print("-", DEV_TURNS_PATH)
print("-", TEST_TURNS_PATH)
print("-", SCHEMA_PATH)
print("-", SUMMARY_PATH)


Saved:
- /content/drive/MyDrive/UTS Courses/Industrial Project/data/evaluation/development_dialogues_v1.jsonl
- /content/drive/MyDrive/UTS Courses/Industrial Project/data/evaluation/heldout_test_dialogues_v1.jsonl
- /content/drive/MyDrive/UTS Courses/Industrial Project/data/evaluation/development_turns_v1.csv
- /content/drive/MyDrive/UTS Courses/Industrial Project/data/evaluation/heldout_test_turns_v1.csv
- /content/drive/MyDrive/UTS Courses/Industrial Project/data/evaluation/preference_state_schema_v1.json
- /content/drive/MyDrive/UTS Courses/Industrial Project/data/evaluation/evaluation_dataset_summary_v1.json


## 16. Reload verification

In [19]:

def read_jsonl(path):
    records = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            records.append(json.loads(line))
    return records


dev_reload = read_jsonl(DEV_DIALOGUES_PATH)
test_reload = read_jsonl(TEST_DIALOGUES_PATH)

assert len(dev_reload) == len(dev_dialogues)
assert len(test_reload) == len(test_dialogues)

print("Reload verification passed.")
print(
    f"Development: {len(dev_reload)} dialogues / {len(dev_turns)} turns"
)
print(
    f"Held-out test: {len(test_reload)} dialogues / {len(test_turns)} turns"
)


Reload verification passed.
Development: 60 dialogues / 155 turns
Held-out test: 130 dialogues / 327 turns



# How these files will be used later

## During local application development

Use:

`development_dialogues_v1.jsonl`

for:

- prompt refinement
- JSON/update-schema debugging
- validator debugging
- correction/removal behaviour
- clarification behaviour
- preference-note handling

It is acceptable to inspect failures and change the implementation.

## During final evaluation

Use:

`heldout_test_dialogues_v1.jsonl`

for the final reported metrics:

- Update Exact Accuracy
- Preference-State Exact Accuracy
- Final-State Accuracy
- Clarification Accuracy
- correction/removal breakdowns
- scenario-type breakdowns

Do not repeatedly tune the prompt against the held-out test after formal evaluation begins.

## Product catalogue

`laptop_product_database_v2.parquet`

remains the complete retrieval corpus. It is **not** split into train/test partitions.

---

After this dataset is generated and sanity-checked, the next engineering stage is:

**refactor the current MVP into Python modules → local FastAPI service → unit/integration tests → Docker → GitHub/AWS CI/CD.**
